# Ellipse Packing Contest — Search Lab

Complete end-to-end baseline for packing 67 identical ellipses for `(2,1)`, `(3,1)`, and `(3,2)`. The solver keeps geometry, search, validation, and serialization separate so we can improve each component strategically.

In [1]:
# ============================================================
# ELLIPSE PACKING CONTEST
# Fast End-to-End Optimizer
#
# Main improvements:
#   1. Numba-compiled objective
#   2. No scipy minimize_scalar inside optimization
#   3. Fast pairwise ellipse approximation during search
#   4. Exact-ish validation only for final candidates
#   5. Block coordinate optimization
#   6. Adaptive W/H shrinking
#   7. Multiple diverse initial layouts
#   8. Random perturbation / repair
#
# Designed for Kaggle CPU environments.
# ============================================================

import math
import random
import time
import re
import os

import numpy as np
import pandas as pd

from dataclasses import dataclass

from scipy.optimize import minimize

try:
    from numba import njit
    NUMBA_AVAILABLE = True
except Exception:
    NUMBA_AVAILABLE = False
    print("WARNING: Numba unavailable. Search will be slower.")


# ============================================================
# 0. CONFIGURATION
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)

N = 67

SHAPES = {
    "2:1": (2.0, 1.0),
    "3:2": (3.0, 2.0),
    "3:1": (3.0, 1.0),
}

# ------------------------------------------------------------
# Search configuration
# ------------------------------------------------------------

# Number of independent initial structures.
N_STARTS = {
    "2:1": 12,
    "3:2": 12,
    "3:1": 16,
}

# Local optimization iterations.
POSITION_STEPS = 180
ANGLE_STEPS = 80
REPAIR_STEPS = 100

# Random perturbation scale.
POSITION_JITTER = 0.08
ANGLE_JITTER = 0.12

# Penalty coefficients.
WALL_PENALTY = 2.0e4
COLLISION_PENALTY = 4.0e4

# Search collision safety margin.
SEARCH_MARGIN = 0.015

# Final numerical tolerance.
FINAL_TOL = 1e-8

# Maximum exact-validation boundary samples.
EXACT_SAMPLES = 48

# Shrinking.
INITIAL_SHRINK = 0.992
MIN_SHRINK = 0.9995

# How many failed squeeze attempts before giving up.
MAX_FAILED_SHRINKS = 10


# ============================================================
# 1. DATA STRUCTURES
# ============================================================

@dataclass
class PackingState:
    W: float
    H: float
    x: np.ndarray
    y: np.ndarray
    theta: np.ndarray

    def copy(self):
        return PackingState(
            float(self.W),
            float(self.H),
            self.x.copy(),
            self.y.copy(),
            self.theta.copy(),
        )

    @property
    def area(self):
        return self.W * self.H


@dataclass
class Geometry:
    a: float
    b: float
    n: int = N

    @property
    def ellipse_area(self):
        return math.pi * self.a * self.b

    @property
    def total_area(self):
        return self.n * self.ellipse_area


# ============================================================
# 2. SCORE
# ============================================================

def density(state, geometry):
    return geometry.total_area / state.area


def compute_kaggle_score(state, geometry):
    d = density(state, geometry)
    return (d * 1000.0) ** 2


# ============================================================
# 3. BASIC GEOMETRY
# ============================================================

def half_extents(a, b, theta):
    c = np.cos(theta)
    s = np.sin(theta)

    rx = np.sqrt(
        a * a * c * c +
        b * b * s * s
    )

    ry = np.sqrt(
        a * a * s * s +
        b * b * c * c
    )

    return rx, ry


def boundary_violation(state, geometry):
    rx, ry = half_extents(
        geometry.a,
        geometry.b,
        state.theta,
    )

    v = (
        np.maximum(0.0, rx - state.x)
        + np.maximum(0.0, state.x + rx - state.W)
        + np.maximum(0.0, ry - state.y)
        + np.maximum(0.0, state.y + ry - state.H)
    )

    return float(np.sum(v))


# ============================================================
# 4. FAST NUMBA COLLISION OBJECTIVE
# ============================================================

if NUMBA_AVAILABLE:

    @njit(cache=True, fastmath=True)
    def fast_overlap_penalty(
        x,
        y,
        theta,
        W,
        H,
        a,
        b,
        margin,
    ):
        """
        Fast differentiability-independent collision penalty.

        Uses directional support radius approximation.

        This is NOT the final geometric validator.
        It is deliberately cheap because it is evaluated
        thousands/millions of times.
        """

        n = len(x)

        penalty = 0.0

        # ----------------------------------------------------
        # Wall penalty
        # ----------------------------------------------------

        for i in range(n):

            c = math.cos(theta[i])
            s = math.sin(theta[i])

            rx = math.sqrt(
                a * a * c * c +
                b * b * s * s
            )

            ry = math.sqrt(
                a * a * s * s +
                b * b * c * c
            )

            left = rx - x[i]
            right = x[i] + rx - W

            bottom = ry - y[i]
            top = y[i] + ry - H

            if left > 0:
                penalty += left * left

            if right > 0:
                penalty += right * right

            if bottom > 0:
                penalty += bottom * bottom

            if top > 0:
                penalty += top * top

        # ----------------------------------------------------
        # Pairwise penalty
        # ----------------------------------------------------

        for i in range(n):

            ci = math.cos(theta[i])
            si = math.sin(theta[i])

            for j in range(i + 1, n):

                dx = x[j] - x[i]
                dy = y[j] - y[i]

                d2 = dx * dx + dy * dy

                # Cheap circular rejection.
                max_r = 2.0 * a + margin

                if d2 > max_r * max_r:
                    continue

                dist = math.sqrt(d2 + 1e-14)

                if dist < 1e-12:
                    penalty += 100.0
                    continue

                phi = math.atan2(dy, dx)

                # Radial support of ellipse i toward j.
                q1 = phi - theta[i]

                c1 = math.cos(q1)
                s1 = math.sin(q1)

                r1 = (
                    a * b /
                    math.sqrt(
                        b * b * c1 * c1 +
                        a * a * s1 * s1 +
                        1e-14
                    )
                )

                # Radial support of ellipse j toward i.
                q2 = phi + math.pi - theta[j]

                c2 = math.cos(q2)
                s2 = math.sin(q2)

                r2 = (
                    a * b /
                    math.sqrt(
                        b * b * c2 * c2 +
                        a * a * s2 * s2 +
                        1e-14
                    )
                )

                required = r1 + r2 + margin

                overlap = required - dist

                if overlap > 0:

                    # Cubic gives little pressure far away
                    # and strong pressure near collision.
                    penalty += overlap * overlap * overlap

        return penalty


    @njit(cache=True, fastmath=True)
    def fast_position_penalty(
        x,
        y,
        theta,
        W,
        H,
        a,
        b,
        margin,
    ):
        """
        Same geometry, but intended for repeated local search.
        """

        return fast_overlap_penalty(
            x,
            y,
            theta,
            W,
            H,
            a,
            b,
            margin,
        )

else:

    def fast_overlap_penalty(
        x,
        y,
        theta,
        W,
        H,
        a,
        b,
        margin,
    ):
        return python_overlap_penalty(
            x, y, theta, W, H, a, b, margin
        )

    fast_position_penalty = fast_overlap_penalty


def python_overlap_penalty(
    x,
    y,
    theta,
    W,
    H,
    a,
    b,
    margin,
):
    n = len(x)
    penalty = 0.0

    c = np.cos(theta)
    s = np.sin(theta)

    rx = np.sqrt(a*a*c*c + b*b*s*s)
    ry = np.sqrt(a*a*s*s + b*b*c*c)

    penalty += np.sum(
        np.maximum(0.0, rx - x) ** 2 +
        np.maximum(0.0, x + rx - W) ** 2 +
        np.maximum(0.0, ry - y) ** 2 +
        np.maximum(0.0, y + ry - H) ** 2
    )

    for i in range(n):

        dx = x[i+1:] - x[i]
        dy = y[i+1:] - y[i]

        dist = np.sqrt(dx*dx + dy*dy + 1e-14)

        phi = np.arctan2(dy, dx)

        q1 = phi - theta[i]

        r1 = (
            a*b /
            np.sqrt(
                b*b*np.cos(q1)**2 +
                a*a*np.sin(q1)**2 +
                1e-14
            )
        )

        q2 = phi + np.pi - theta[i+1:]

        r2 = (
            a*b /
            np.sqrt(
                b*b*np.cos(q2)**2 +
                a*a*np.sin(q2)**2 +
                1e-14
            )
        )

        overlap = np.maximum(
            0.0,
            r1 + r2 + margin - dist
        )

        penalty += np.sum(overlap ** 3)

    return float(penalty)


# ============================================================
# 5. FAST OBJECTIVE
# ============================================================

def search_objective(
    params,
    geometry,
    W,
    H,
):
    n = geometry.n

    x = params[:n]
    y = params[n:2*n]
    theta = params[2*n:]

    penalty = fast_overlap_penalty(
        x,
        y,
        theta,
        W,
        H,
        geometry.a,
        geometry.b,
        SEARCH_MARGIN,
    )

    # Tiny orientation regularizer.
    # Prevents meaningless angle explosions.
    angle_reg = 1e-7 * np.sum(
        np.sin(2.0 * theta) ** 2
    )

    return (
        WALL_PENALTY * penalty
        + angle_reg
    )


# ============================================================
# 6. CONSTRUCTION UTILITIES
# ============================================================

def minimum_safe_spacing(geometry):
    return 2.0 * geometry.a


def make_rectangle_for_density(
    geometry,
    target_density=0.78,
    aspect_ratio=1.35,
):
    """
    Creates a reasonably compact initial rectangle.

    We deliberately start much closer to the useful density
    region than the original 0.65 initializer.
    """

    area = geometry.total_area / target_density

    W = math.sqrt(
        area * aspect_ratio
    )

    H = area / W

    return W, H


def clip_state(state, geometry):
    rx, ry = half_extents(
        geometry.a,
        geometry.b,
        state.theta,
    )

    state.x = np.clip(
        state.x,
        rx,
        state.W - rx,
    )

    state.y = np.clip(
        state.y,
        ry,
        state.H - ry,
    )

    return state


# ============================================================
# 7. GRID INITIALIZER
# ============================================================

def grid_initializer(
    geometry,
    rows,
    cols,
    angle_mode="zero",
    jitter=0.0,
):
    W, H = make_rectangle_for_density(
        geometry,
        target_density=0.76,
        aspect_ratio=max(
            1.05,
            cols / rows
        ),
    )

    rx_min = geometry.a
    ry_min = geometry.b

    xs = np.linspace(
        rx_min,
        W - rx_min,
        cols,
    )

    ys = np.linspace(
        ry_min,
        H - ry_min,
        rows,
    )

    x = []
    y = []

    for r in range(rows):

        offset = 0.0

        if r % 2 == 1:
            offset = (
                (xs[1] - xs[0])
                * 0.45
                if len(xs) > 1
                else 0.0
            )

        for c in range(cols):

            if len(x) >= geometry.n:
                break

            xx = xs[c] + offset

            # Wrap shifted points back inside.
            if xx > W - rx_min:
                xx -= 0.5 * (xs[1] - xs[0])

            x.append(xx)
            y.append(ys[r])

    x = np.asarray(x, dtype=np.float64)
    y = np.asarray(y, dtype=np.float64)

    # Ensure exactly N.
    x = x[:geometry.n]
    y = y[:geometry.n]

    if angle_mode == "zero":
        theta = np.zeros(geometry.n)

    elif angle_mode == "alternate":
        theta = np.where(
            np.arange(geometry.n) % 2 == 0,
            0.0,
            np.pi / 2,
        )

    elif angle_mode == "random":
        theta = np.random.uniform(
            0,
            np.pi,
            geometry.n,
        )

    else:
        theta = np.zeros(geometry.n)

    if jitter > 0:
        x += np.random.normal(
            0,
            jitter,
            geometry.n,
        )

        y += np.random.normal(
            0,
            jitter,
            geometry.n,
        )

        theta += np.random.normal(
            0,
            jitter * 0.5,
            geometry.n,
        )

    state = PackingState(
        W,
        H,
        x,
        y,
        theta,
    )

    return clip_state(
        state,
        geometry,
    )


# ============================================================
# 8. STRIPE INITIALIZER
# ============================================================

def stripe_initializer(
    geometry,
    horizontal=True,
    alternating=False,
):
    """
    Particularly useful for elongated 3:1 ellipses.
    """

    if horizontal:

        rows = max(
            5,
            int(
                math.sqrt(
                    geometry.n *
                    geometry.b /
                    geometry.a
                )
            )
        )

        cols = math.ceil(
            geometry.n / rows
        )

    else:

        cols = max(
            5,
            int(
                math.sqrt(
                    geometry.n *
                    geometry.b /
                    geometry.a
                )
            )
        )

        rows = math.ceil(
            geometry.n / cols
        )

    state = grid_initializer(
        geometry,
        rows,
        cols,
        angle_mode="zero",
    )

    if horizontal:

        state.theta[:] = 0.0

    else:

        state.theta[:] = np.pi / 2

    if alternating:

        for r in range(rows):

            start = r * cols
            end = min(
                (r + 1) * cols,
                geometry.n,
            )

            if r % 2 == 1:
                state.theta[
                    start:end
                ] += np.pi / 2

    return clip_state(
        state,
        geometry,
    )


# ============================================================
# 9. RANDOMIZED INITIALIZER
# ============================================================

def randomized_initializer(
    geometry,
    base,
    position_noise=0.25,
    angle_noise=0.25,
):
    state = base.copy()

    state.x += np.random.normal(
        0,
        position_noise,
        geometry.n,
    )

    state.y += np.random.normal(
        0,
        position_noise,
        geometry.n,
    )

    state.theta += np.random.normal(
        0,
        angle_noise,
        geometry.n,
    )

    state.theta %= np.pi

    return clip_state(
        state,
        geometry,
    )


# ============================================================
# 10. GENERATE DIVERSE STARTS
# ============================================================

def generate_initial_states(geometry):
    states = []

    # --------------------------------------------------------
    # Square-ish grids
    # --------------------------------------------------------

    grid_candidates = [
        (8, 9),
        (9, 8),
        (7, 10),
        (10, 7),
        (6, 12),
        (12, 6),
    ]

    for rows, cols in grid_candidates:

        states.append(
            grid_initializer(
                geometry,
                rows,
                cols,
                angle_mode="zero",
                jitter=0.0,
            )
        )

        states.append(
            grid_initializer(
                geometry,
                rows,
                cols,
                angle_mode="alternate",
                jitter=0.0,
            )
        )

    # --------------------------------------------------------
    # Stripe layouts
    # --------------------------------------------------------

    states.append(
        stripe_initializer(
            geometry,
            horizontal=True,
            alternating=False,
        )
    )

    states.append(
        stripe_initializer(
            geometry,
            horizontal=False,
            alternating=False,
        )
    )

    # --------------------------------------------------------
    # Randomized versions
    # --------------------------------------------------------

    base = states[0]

    for _ in range(12):

        states.append(
            randomized_initializer(
                geometry,
                base,
                position_noise=0.18,
                angle_noise=0.20,
            )
        )

    # --------------------------------------------------------
    # Keep enough diversity but don't explode runtime.
    # --------------------------------------------------------

    random.shuffle(states)

    return states


# ============================================================
# 11. FAST LOCAL REPAIR
# ============================================================

def repair_positions(
    state,
    geometry,
    iterations=100,
    step_size=0.08,
):
    """
    Cheap physics-like collision repair.

    Instead of invoking scipy for every repair step,
    directly push nearby ellipses apart.
    """

    s = state.copy()

    for iteration in range(iterations):

        rx, ry = half_extents(
            geometry.a,
            geometry.b,
            s.theta,
        )

        moved = False

        for i in range(geometry.n):

            for j in range(i + 1, geometry.n):

                dx = s.x[j] - s.x[i]
                dy = s.y[j] - s.y[i]

                dist = math.sqrt(
                    dx * dx +
                    dy * dy +
                    1e-12
                )

                # Approximate radial requirement.
                phi = math.atan2(
                    dy,
                    dx,
                )

                r1 = (
                    geometry.a *
                    geometry.b
                ) / math.sqrt(
                    (
                        geometry.b *
                        math.cos(
                            phi - s.theta[i]
                        )
                    ) ** 2
                    +
                    (
                        geometry.a *
                        math.sin(
                            phi - s.theta[i]
                        )
                    ) ** 2
                    +
                    1e-12
                )

                r2 = (
                    geometry.a *
                    geometry.b
                ) / math.sqrt(
                    (
                        geometry.b *
                        math.cos(
                            phi + math.pi -
                            s.theta[j]
                        )
                    ) ** 2
                    +
                    (
                        geometry.a *
                        math.sin(
                            phi + math.pi -
                            s.theta[j]
                        )
                    ) ** 2
                    +
                    1e-12
                )

                required = (
                    r1 +
                    r2 +
                    SEARCH_MARGIN
                )

                overlap = (
                    required -
                    dist
                )

                if overlap <= 0:
                    continue

                moved = True

                ux = dx / dist
                uy = dy / dist

                push = (
                    overlap *
                    step_size
                )

                s.x[i] -= ux * push
                s.y[i] -= uy * push

                s.x[j] += ux * push
                s.y[j] += uy * push

        s = clip_state(
            s,
            geometry,
        )

        if not moved:
            break

        step_size *= 0.985

    return s


# ============================================================
# 12. SCIPY LOCAL POSITION OPTIMIZATION
# ============================================================

def optimize_positions(
    state,
    geometry,
    maxiter=POSITION_STEPS,
):
    """
    Optimize x/y while keeping angles fixed.

    This is substantially smaller than optimizing all
    201 variables simultaneously.
    """

    n = geometry.n

    initial = np.concatenate([
        state.x,
        state.y,
    ])

    bounds = (
        [(0.0, state.W)] * n
        +
        [(0.0, state.H)] * n
    )

    def objective(p):

        x = p[:n]
        y = p[n:]

        return fast_position_penalty(
            x,
            y,
            state.theta,
            state.W,
            state.H,
            geometry.a,
            geometry.b,
            SEARCH_MARGIN,
        )

    result = minimize(
        objective,
        initial,
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": maxiter,
            "ftol": 1e-8,
            "gtol": 1e-6,
            "maxls": 20,
        },
    )

    state.x = result.x[:n]
    state.y = result.x[n:]

    return clip_state(
        state,
        geometry,
    )


# ============================================================
# 13. ANGLE OPTIMIZATION
# ============================================================

def optimize_angles(
    state,
    geometry,
    maxiter=ANGLE_STEPS,
):
    """
    Optimize only orientations.

    Positions remain fixed.
    """

    n = geometry.n

    initial = state.theta.copy()

    bounds = [
        (0.0, math.pi)
        for _ in range(n)
    ]

    def objective(theta):

        return fast_overlap_penalty(
            state.x,
            state.y,
            theta,
            state.W,
            state.H,
            geometry.a,
            geometry.b,
            SEARCH_MARGIN,
        )

    result = minimize(
        objective,
        initial,
        method="L-BFGS-B",
        bounds=bounds,
        options={
            "maxiter": maxiter,
            "ftol": 1e-8,
            "gtol": 1e-6,
            "maxls": 20,
        },
    )

    state.theta = result.x

    return clip_state(
        state,
        geometry,
    )


# ============================================================
# 14. LOCAL IMPROVEMENT LOOP
# ============================================================

def relax_state(
    state,
    geometry,
    rounds=3,
):
    best = state.copy()

    best_penalty = fast_overlap_penalty(
        best.x,
        best.y,
        best.theta,
        best.W,
        best.H,
        geometry.a,
        geometry.b,
        SEARCH_MARGIN,
    )

    for _ in range(rounds):

        candidate = best.copy()

        candidate = optimize_positions(
            candidate,
            geometry,
        )

        candidate = repair_positions(
            candidate,
            geometry,
            iterations=REPAIR_STEPS,
        )

        candidate = optimize_angles(
            candidate,
            geometry,
        )

        candidate = repair_positions(
            candidate,
            geometry,
            iterations=50,
        )

        penalty = fast_overlap_penalty(
            candidate.x,
            candidate.y,
            candidate.theta,
            candidate.W,
            candidate.H,
            geometry.a,
            geometry.b,
            SEARCH_MARGIN,
        )

        if penalty < best_penalty:

            best = candidate
            best_penalty = penalty

        else:
            break

    return best


# ============================================================
# 15. EXACT-ish FINAL GEOMETRY
# ============================================================

def point_q(
    px,
    py,
    cx,
    cy,
    a,
    b,
    theta,
):
    dx = px - cx
    dy = py - cy

    c = math.cos(theta)
    s = math.sin(theta)

    u = c * dx + s * dy
    v = -s * dx + c * dy

    return (
        (u / a) ** 2 +
        (v / b) ** 2
    )


def ellipse_boundary_point(
    cx,
    cy,
    a,
    b,
    theta,
    z,
):
    c = math.cos(theta)
    s = math.sin(theta)

    cz = math.cos(z)
    sz = math.sin(z)

    return (
        cx +
        c * a * cz -
        s * b * sz,

        cy +
        s * a * cz +
        c * b * sz,
    )


def exact_pair_check(
    i,
    j,
    state,
    geometry,
    samples=EXACT_SAMPLES,
    tol=FINAL_TOL,
):
    """
    More expensive geometric check.

    Used ONLY during final validation.

    We first perform cheap AABB rejection.
    """

    a = geometry.a
    b = geometry.b

    xi = float(state.x[i])
    yi = float(state.y[i])
    ti = float(state.theta[i])

    xj = float(state.x[j])
    yj = float(state.y[j])
    tj = float(state.theta[j])

    rxi, ryi = half_extents(
        a,
        b,
        np.array([ti])
    )

    rxj, ryj = half_extents(
        a,
        b,
        np.array([tj])
    )

    rxi = float(rxi[0])
    ryi = float(ryi[0])
    rxj = float(rxj[0])
    ryj = float(ryj[0])

    # --------------------------------------------------------
    # AABB rejection
    # --------------------------------------------------------

    if (
        xi + rxi <= xj - rxj - tol
        or
        xj + rxj <= xi - rxi - tol
        or
        yi + ryi <= yj - ryj - tol
        or
        yj + ryj <= yi - ryi - tol
    ):
        return False

    # --------------------------------------------------------
    # Center containment
    # --------------------------------------------------------

    if point_q(
        xi,
        yi,
        xj,
        yj,
        a,
        b,
        tj,
    ) <= 1.0 + tol:

        return True

    if point_q(
        xj,
        yj,
        xi,
        yi,
        a,
        b,
        ti,
    ) <= 1.0 + tol:

        return True

    # --------------------------------------------------------
    # Boundary sampling
    # --------------------------------------------------------

    angles = np.linspace(
        0.0,
        2.0 * math.pi,
        samples,
        endpoint=False,
    )

    for z in angles:

        px, py = ellipse_boundary_point(
            xi,
            yi,
            a,
            b,
            ti,
            float(z),
        )

        if point_q(
            px,
            py,
            xj,
            yj,
            a,
            b,
            tj,
        ) <= 1.0 + tol:

            return True

        px, py = ellipse_boundary_point(
            xj,
            yj,
            a,
            b,
            tj,
            float(z),
        )

        if point_q(
            px,
            py,
            xi,
            yi,
            a,
            b,
            ti,
        ) <= 1.0 + tol:

            return True

    return False


# ============================================================
# 16. FINAL COLLISION VALIDATOR
# ============================================================

def collision_count_exact(
    state,
    geometry,
):
    rx, ry = half_extents(
        geometry.a,
        geometry.b,
        state.theta,
    )

    xmin = state.x - rx
    xmax = state.x + rx
    ymin = state.y - ry
    ymax = state.y + ry

    count = 0

    for i in range(geometry.n):

        for j in range(i + 1, geometry.n):

            # Very cheap AABB rejection.
            if (
                xmax[i] <= xmin[j]
                or
                xmax[j] <= xmin[i]
                or
                ymax[i] <= ymin[j]
                or
                ymax[j] <= ymin[i]
            ):
                continue

            if exact_pair_check(
                i,
                j,
                state,
                geometry,
            ):
                count += 1

    return count


def feasible_exact(
    state,
    geometry,
):
    if state.W <= 0 or state.H <= 0:
        return False

    if len(state.x) != geometry.n:
        return False

    if not np.all(np.isfinite(state.x)):
        return False

    if not np.all(np.isfinite(state.y)):
        return False

    if not np.all(np.isfinite(state.theta)):
        return False

    if boundary_violation(
        state,
        geometry,
    ) > FINAL_TOL:
        return False

    return (
        collision_count_exact(
            state,
            geometry,
        )
        == 0
    )


# ============================================================
# 17. ADAPTIVE RECTANGLE SHRINK
# ============================================================

def try_shrink(
    state,
    geometry,
    dimension,
    factor,
):
    candidate = state.copy()

    if dimension == "W":

        new_W = candidate.W * factor

        if new_W <= 2.0 * geometry.a:
            return None

        candidate.W = new_W

    else:

        new_H = candidate.H * factor

        if new_H <= 2.0 * geometry.b:
            return None

        candidate.H = new_H

    candidate = clip_state(
        candidate,
        geometry,
    )

    # Fast repair.
    candidate = repair_positions(
        candidate,
        geometry,
        iterations=80,
        step_size=0.12,
    )

    # Local position refinement.
    candidate = optimize_positions(
        candidate,
        geometry,
        maxiter=100,
    )

    # Angle refinement.
    candidate = optimize_angles(
        candidate,
        geometry,
        maxiter=40,
    )

    candidate = repair_positions(
        candidate,
        geometry,
        iterations=60,
        step_size=0.10,
    )

    return candidate


def adaptive_shrink(
    state,
    geometry,
):
    """
    Alternates width and height reduction.

    If one direction is more productive, it gets more attempts.
    """

    best = state.copy()

    factor = INITIAL_SHRINK

    failed = {
        "W": 0,
        "H": 0,
    }

    iteration = 0

    while iteration < 100:

        iteration += 1

        candidates = []

        for dim in ("W", "H"):

            if failed[dim] >= MAX_FAILED_SHRINKS:
                continue

            candidate = try_shrink(
                best,
                geometry,
                dim,
                factor,
            )

            if candidate is None:
                continue

            fast_penalty = fast_overlap_penalty(
                candidate.x,
                candidate.y,
                candidate.theta,
                candidate.W,
                candidate.H,
                geometry.a,
                geometry.b,
                SEARCH_MARGIN,
            )

            candidates.append(
                (
                    candidate,
                    fast_penalty,
                    dim,
                )
            )

        if not candidates:

            break

        # ----------------------------------------------------
        # Prefer lower area, then lower collision penalty.
        # ----------------------------------------------------

        candidates.sort(
            key=lambda z: (
                z[0].area,
                z[1],
            )
        )

        candidate, penalty, dim = candidates[0]

        # Fast feasibility threshold.
        if penalty < 1e-8:

            best = candidate
            failed[dim] = 0

            # Try to continue aggressively.
            factor = max(
                MIN_SHRINK,
                factor,
            )

        else:

            failed[dim] += 1

            # If both directions struggle, make the shrink
            # smaller rather than immediately abandoning.
            factor = min(
                MIN_SHRINK,
                factor + 0.001,
            )

        # ----------------------------------------------------
        # Occasionally perform an expensive validation.
        # ----------------------------------------------------

        if iteration % 5 == 0:

            if feasible_exact(
                best,
                geometry,
            ):

                print(
                    f"      shrink {iteration:03d} "
                    f"W={best.W:.6f} "
                    f"H={best.H:.6f} "
                    f"density={density(best, geometry):.6f}"
                )

    return best


# ============================================================
# 18. FINAL POLISH
# ============================================================

def polish(
    state,
    geometry,
):
    best = state.copy()

    # Several small local passes.
    for _ in range(3):

        best = optimize_positions(
            best,
            geometry,
            maxiter=120,
        )

        best = optimize_angles(
            best,
            geometry,
            maxiter=50,
        )

        best = repair_positions(
            best,
            geometry,
            iterations=80,
            step_size=0.08,
        )

    return best


# ============================================================
# 19. SOLVE ONE SHAPE
# ============================================================

def solve_shape(
    name,
    n_starts=None,
):
    geometry = Geometry(
        *SHAPES[name]
    )

    if n_starts is None:
        n_starts = N_STARTS[name]

    print()
    print("=" * 78)
    print(f"SOLVING {name}   a={geometry.a} b={geometry.b}")
    print("=" * 78)

    initial_states = generate_initial_states(
        geometry
    )

    # Randomly choose a subset.
    if len(initial_states) > n_starts:

        random.shuffle(
            initial_states
        )

        initial_states = initial_states[
            :n_starts
        ]

    best = None

    for start_idx, initial in enumerate(
        initial_states
    ):

        t0 = time.time()

        print(
            f"\n[{name}] START "
            f"{start_idx + 1}/{len(initial_states)}"
        )

        state = initial.copy()

        # ----------------------------------------------------
        # Initial repair.
        # ----------------------------------------------------

        state = repair_positions(
            state,
            geometry,
            iterations=100,
            step_size=0.12,
        )

        # ----------------------------------------------------
        # Local optimization.
        # ----------------------------------------------------

        state = relax_state(
            state,
            geometry,
            rounds=3,
        )

        # ----------------------------------------------------
        # Shrink rectangle.
        # ----------------------------------------------------

        state = adaptive_shrink(
            state,
            geometry,
        )

        # ----------------------------------------------------
        # Final local polish.
        # ----------------------------------------------------

        state = polish(
            state,
            geometry,
        )

        # ----------------------------------------------------
        # Exact validation.
        # ----------------------------------------------------

        ok = feasible_exact(
            state,
            geometry,
        )

        d = density(
            state,
            geometry,
        )

        score = compute_kaggle_score(
            state,
            geometry,
        )

        elapsed = time.time() - t0

        print(
            f"[{name}] "
            f"D={d:.8f} "
            f"Score={score:,.0f} "
            f"W={state.W:.6f} "
            f"H={state.H:.6f} "
            f"Feasible={ok} "
            f"Time={elapsed:.1f}s"
        )

        # ----------------------------------------------------
        # Keep best exact-feasible candidate.
        # ----------------------------------------------------

        if ok:

            if (
                best is None
                or
                state.area < best.area
            ):

                best = state.copy()

                print(
                    f"      NEW BEST → "
                    f"density={d:.8f}"
                )

    if best is None:
        raise RuntimeError(
            f"No feasible solution found for {name}"
        )

    print()
    print(
        f"BEST {name}: "
        f"density={density(best, geometry):.8f} "
        f"score={compute_kaggle_score(best, geometry):,.0f}"
    )

    return best


# ============================================================
# 20. SERIALIZATION
# ============================================================

def fmt10_safe(value):
    """
    Competition uses first 10 decimal digits.
    """

    value = float(value)

    s = f"{value:.14f}"

    if "." not in s:
        return s

    integer, decimal = s.split(".")

    return (
        integer +
        "." +
        decimal[:10]
    )


def state_to_block(
    name,
    state,
):
    """
    IMPORTANT:
    Uses the structure from the competition description:

        ratio: 2:1
        W*H
        000: x y rotation
        ...

    """

    lines = [
        f"ratio: {name}",
        (
            f"{fmt10_safe(state.W)}*"
            f"{fmt10_safe(state.H)}"
        ),
    ]

    for i in range(N):

        degrees = (
            np.degrees(
                state.theta[i]
            )
            % 360.0
        )

        lines.append(
            f"{i:03d}: "
            f"{fmt10_safe(state.x[i])} "
            f"{fmt10_safe(state.y[i])} "
            f"{fmt10_safe(degrees)}"
        )

    return "\n".join(lines)


def build_submission_text(
    best_states,
):
    # Competition order.
    order = [
        "2:1",
        "3:2",
        "3:1",
    ]

    return "\n".join(
        state_to_block(
            name,
            best_states[name],
        )
        for name in order
    )


def build_submission_csv(
    best_states,
    path="submission.csv",
):
    text = build_submission_text(
        best_states
    )

    df = pd.DataFrame({
        "row_id": [0],
        "submission_text": [text],
    })

    df.to_csv(
        path,
        index=False,
    )

    return df


# ============================================================
# 21. ROUND-TRIP PARSER
# ============================================================

def parse_submission_text(text):

    blocks = {}

    lines = [
        x.strip()
        for x in str(text).splitlines()
        if x.strip()
    ]

    i = 0

    while i < len(lines):

        ratio_match = re.match(
            r"^ratio:\s*(2:1|3:2|3:1)$",
            lines[i],
        )

        if not ratio_match:
            raise ValueError(
                f"Invalid ratio line: {lines[i]}"
            )

        name = ratio_match.group(1)

        i += 1

        wh = re.match(
            r"^([-+0-9.eE]+)\*([-+0-9.eE]+)$",
            lines[i],
        )

        if not wh:
            raise ValueError(
                f"Invalid W*H line: {lines[i]}"
            )

        W = float(
            wh.group(1)
        )

        H = float(
            wh.group(2)
        )

        i += 1

        xs = []
        ys = []
        theta = []

        for expected in range(N):

            if i >= len(lines):
                raise ValueError(
                    "Unexpected end of block"
                )

            m = re.match(
                r"^(\d{3}):\s*"
                r"([-+0-9.eE]+)\s+"
                r"([-+0-9.eE]+)\s+"
                r"([-+0-9.eE]+)$",
                lines[i],
            )

            if not m:
                raise ValueError(
                    f"Invalid row: {lines[i]}"
                )

            idx = int(m.group(1))

            if idx != expected:
                raise ValueError(
                    f"Expected {expected:03d}, got {idx:03d}"
                )

            xs.append(
                float(m.group(2))
            )

            ys.append(
                float(m.group(3))
            )

            theta.append(
                math.radians(
                    float(m.group(4))
                )
            )

            i += 1

        blocks[name] = PackingState(
            W,
            H,
            np.asarray(
                xs,
                dtype=np.float64,
            ),
            np.asarray(
                ys,
                dtype=np.float64,
            ),
            np.asarray(
                theta,
                dtype=np.float64,
            ),
        )

    if set(blocks) != set(SHAPES):

        raise ValueError(
            f"Missing shapes: "
            f"{set(SHAPES) - set(blocks)}"
        )

    return blocks


# ============================================================
# 22. STRICT VALIDATION
# ============================================================

def strict_validate(
    state,
    geometry,
):
    assert state.W > 0
    assert state.H > 0

    assert len(state.x) == N
    assert len(state.y) == N
    assert len(state.theta) == N

    assert np.all(
        np.isfinite(state.x)
    )

    assert np.all(
        np.isfinite(state.y)
    )

    assert np.all(
        np.isfinite(state.theta)
    )

    bv = boundary_violation(
        state,
        geometry,
    )

    assert bv <= FINAL_TOL, (
        f"Boundary violation={bv}"
    )

    collisions = collision_count_exact(
        state,
        geometry,
    )

    assert collisions == 0, (
        f"{collisions} ellipse collisions"
    )

    return True


# ============================================================
# 23. VISUALIZATION
# ============================================================

def plot_packing(
    state,
    geometry,
    title=None,
):
    import matplotlib.pyplot as plt
    from matplotlib.patches import Ellipse as MplEllipse

    fig, ax = plt.subplots(
        figsize=(12, 8)
    )

    ax.set_xlim(
        0,
        state.W,
    )

    ax.set_ylim(
        0,
        state.H,
    )

    for i in range(N):

        e = MplEllipse(
            (
                state.x[i],
                state.y[i],
            ),
            width=2 * geometry.a,
            height=2 * geometry.b,
            angle=np.degrees(
                state.theta[i]
            ),
            fill=False,
            linewidth=0.8,
        )

        ax.add_patch(e)

        ax.text(
            state.x[i],
            state.y[i],
            str(i),
            fontsize=6,
            ha="center",
            va="center",
        )

    ax.set_aspect(
        "equal",
        adjustable="box",
    )

    ax.set_title(
        title or
        f"{geometry.a}:{geometry.b}"
    )

    ax.grid(
        alpha=0.2
    )

    plt.show()


# ============================================================
# 24. MAIN EXECUTION
# ============================================================

if __name__ == "__main__":

    total_start = time.time()

    print("=" * 78)
    print("ELLIPSE PACKING OPTIMIZER")
    print("=" * 78)

    print(
        f"Numba available: "
        f"{NUMBA_AVAILABLE}"
    )

    print(
        f"Number of ellipses: {N}"
    )

    print(
        "Target density for 750k: "
        f"{math.sqrt(750000) / 1000:.6f}"
    )

    print(
        "Target density for 770k: "
        f"{math.sqrt(770000) / 1000:.6f}"
    )

    # --------------------------------------------------------
    # Solve each shape independently.
    # --------------------------------------------------------

    best_states = {}

    # Solve hardest elongated geometry first.
    solve_order = [
        "3:1",
        "2:1",
        "3:2",
    ]

    for name in solve_order:

        best_states[name] = solve_shape(
            name,
            n_starts=N_STARTS[name],
        )

    # --------------------------------------------------------
    # Final validation.
    # --------------------------------------------------------

    print()
    print("=" * 78)
    print("FINAL EXACT VALIDATION")
    print("=" * 78)

    summary = []

    for name in [
        "2:1",
        "3:2",
        "3:1",
    ]:

        state = best_states[name]

        geometry = Geometry(
            *SHAPES[name]
        )

        print(
            f"\nValidating {name}..."
        )

        strict_validate(
            state,
            geometry,
        )

        d = density(
            state,
            geometry,
        )

        score = compute_kaggle_score(
            state,
            geometry,
        )

        summary.append({
            "shape": name,
            "W": state.W,
            "H": state.H,
            "area": state.area,
            "density": d,
            "score": score,
            "feasible": True,
        })

        print(
            f"  W       = {state.W:.10f}"
        )

        print(
            f"  H       = {state.H:.10f}"
        )

        print(
            f"  Area    = {state.area:.10f}"
        )

        print(
            f"  Density = {d:.10f}"
        )

        print(
            f"  Score   = {score:,.2f}"
        )

    # --------------------------------------------------------
    # Overall score.
    # --------------------------------------------------------

    df_summary = pd.DataFrame(
        summary
    )

    final_score = (
        df_summary["score"].mean()
    )

    print()
    print("=" * 78)
    print("FINAL RESULTS")
    print("=" * 78)

    print(
        df_summary.to_string(
            index=False
        )
    )

    print()
    print(
        f"OVERALL KAGGLE SCORE: "
        f"{final_score:,.2f}"
    )

    print(
        f"TOTAL RUNTIME: "
        f"{time.time() - total_start:.1f}s"
    )

    # --------------------------------------------------------
    # Submission.
    # --------------------------------------------------------

    submission = build_submission_csv(
        best_states,
        "submission.csv",
    )

    print()
    print(
        "Created submission.csv"
    )

    # --------------------------------------------------------
    # Round-trip test.
    # --------------------------------------------------------

    parsed = parse_submission_text(
        submission.loc[
            0,
            "submission_text"
        ]
    )

    for name, state in parsed.items():

        geometry = Geometry(
            *SHAPES[name]
        )

        strict_validate(
            state,
            geometry,
        )

    print(
        "ROUND-TRIP VALIDATION PASSED"
    )

    print(
        "Ready for Kaggle submission."
    )

ELLIPSE PACKING OPTIMIZER
Numba available: True
Number of ellipses: 67
Target density for 750k: 0.866025
Target density for 770k: 0.877496

SOLVING 3:1   a=3.0 b=1.0

[3:1] START 1/16
[3:1] D=0.76727938 Score=588,718 W=40.418076 H=20.361828 Feasible=False Time=43.8s

[3:1] START 2/16
[3:1] D=0.76958525 Score=592,261 W=29.433398 H=27.877177 Feasible=False Time=48.3s

[3:1] START 3/16
[3:1] D=0.76843145 Score=590,487 W=30.268109 H=27.149104 Feasible=False Time=47.3s

[3:1] START 4/16
[3:1] D=0.76000000 Score=577,600 W=29.536622 H=28.130116 Feasible=False Time=69.7s

[3:1] START 5/16
[3:1] D=0.76000000 Score=577,600 W=34.452215 H=24.116551 Feasible=False Time=77.4s

[3:1] START 6/16
[3:1] D=0.76000000 Score=577,600 W=29.536622 H=28.130116 Feasible=False Time=57.1s

[3:1] START 7/16
      shrink 005 W=29.536622 H=28.130116 density=0.760000
      shrink 010 W=29.536622 H=28.130116 density=0.760000
      shrink 015 W=29.536622 H=28.130116 density=0.760000
      shrink 020 W=29.536622 H=28.13